### Data Transformation

In [1]:
import os
%pwd

'/workspaces/summarizer/textsummariser/research'

In [2]:
os.chdir("../")

In [3]:
%pwd

'/workspaces/summarizer/textsummariser'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [5]:
from src.textSummariser.constants import *
from src.textSummariser.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(self,
                config_path = CONFIG_FILE_PATH,
                params_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = Path(config.root_dir),
            data_path = Path(config.data_path),
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

In [7]:
import os
from src.textSummariser.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk

## DataTransformation Components

In [8]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_examples_to_features(self, example_batch): # Added self here!
        
        input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)

        target_encodings = self.tokenizer(text_target=example_batch['summary'], max_length=128, truncation=True)

        return {
            'input_ids' : input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        
        # FIXED: Glued the paths together into one string
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))

In [9]:
config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.convert()


[2026-09-27 15:14:47,313: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-09-27 15:14:47,339: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-27 15:14:47,342: INFO: common: created directory at: artifacts]
[2026-09-27 15:14:47,358: INFO: common: created directory at: artifacts/data_transformation]


[2026-09-27 15:14:47,500: INFO: _client: HTTP Request: HEAD https://huggingface.co/sshleifer/distilbart-cnn-12-6/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]


[2026-09-27 15:14:47,501: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-09-27 15:14:47,508: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sshleifer/distilbart-cnn-12-6/a4f8f3ea906ed274767e9906dbaede7531d660ff/config.json?%2Fsshleifer%2Fdistilbart-cnn-12-6%2Fresolve%2Fmain%2Fconfig.json=&etag=%22fade82e22ec4f4b61f6f2eb3986ce4a64a8e23df%22 "HTTP/1.1 200 OK"]
[2026-09-27 15:14:47,693: INFO: _client: HTTP Request: HEAD https://huggingface.co/sshleifer/distilbart-cnn-12-6/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-09-27 15:14:47,701: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sshleifer/distilbart-cnn-12-6/a4f8f3ea906ed274767e9906dbaede7531d660ff/tokenizer_config.json?%2Fsshleifer%2Fdistilbart-cnn-12-6%2Fresolve%2Fmain%2Ftokenizer_config.json=&etag=%22be4d21d94f3b4687e5a54

Saving the dataset (0/1 shards):   0%|          | 0/14732 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/819 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]